<a href="https://colab.research.google.com/github/xinghanchen-xc/DATA230-Final-Group-Project/blob/main/High_Performance_Data_Engineering_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# -----------------------------------------------------------------------------
# PROJECT: STUDENT WELL-BEING ANALYTICS (1M RECORDS)
# COMPONENT: TASK 1 - SCALABLE DATA ENGINEERING PIPELINE
# ENVIRONMENT: GOOGLE COLAB (T4 GPU)
# -----------------------------------------------------------------------------

import kagglehub
import cudf
import pandas as pd
import time
import os
from google.colab import drive, files

def run_data_engineering_pipeline():
    # 1. GOOGLE DRIVE MOUNTING
    # -------------------------------------------------------------------------
    print("Mounting Google Drive for persistent storage...")
    drive.mount('/content/drive', force_remount=True)
    STORAGE_PATH = "/content/drive/MyDrive/Colab Notebooks/"

    # 2. DATA ACQUISITION VIA KAGGLEHUB API
    # -------------------------------------------------------------------------
    print("Initiating data acquisition protocol from Kaggle...")
    try:
        path = kagglehub.dataset_download("sharmajicoder/student-mental-health-and-burnout")
        csv_file = os.path.join(path, "student_mental_health_burnout_1M.csv")
    except Exception as e:
        print(f"Data acquisition failed: {e}")
        return

    # 3. COMPUTATIONAL BENCHMARKING: CPU (PANDAS) VS. GPU (CUDF)
    # -------------------------------------------------------------------------
    print("Executing performance benchmarking: CPU (Pandas) vs. GPU (cuDF)...")

    # Latency: CPU Standard Implementation
    start_cpu = time.time()
    pdf = pd.read_csv(csv_file)
    cpu_latency = time.time() - start_cpu

    # Latency: GPU Accelerated Implementation
    start_gpu = time.time()
    gdf = cudf.read_csv(csv_file)
    gpu_latency = time.time() - start_gpu

    throughput_gain = cpu_latency / gpu_latency
    print(f"CPU Processing Latency: {cpu_latency:.4f} seconds")
    print(f"GPU Processing Latency: {gpu_latency:.4f} seconds")
    print(f"Hardware Acceleration Throughput Gain: {throughput_gain:.2f}x")

    # 4. HIGH-FIDELITY FEATURE ENGINEERING (N=1,000,000)
    # -------------------------------------------------------------------------
    print("Commencing high-fidelity feature engineering on 1,000,000 records...")
    gdf.columns = gdf.columns.str.strip().str.lower()

    # Domain-specific interaction metrics construction
    gdf['academic_resilience'] = gdf['exam_pressure'] / (gdf['sleep_hours'] + 1)
    gdf['study_sleep_gap'] = gdf['study_hours_per_day'] - gdf['sleep_hours']
    gdf['socio_economic_multiplier'] = gdf['financial_stress'] * gdf['exam_pressure']
    gdf['effort_reward_index'] = gdf['study_hours_per_day'] / (gdf['academic_performance'] + 0.1)
    gdf['total_cognitive_load'] = gdf['study_hours_per_day'] + gdf['screen_time']
    gdf['anxiety_pressure_impact'] = gdf['anxiety_score'] * gdf['exam_pressure']
    gdf['social_support_ratio'] = gdf['social_support'] / (gdf['family_expectation'] + 1)
    gdf['combined_distress_index'] = gdf['depression_score'] + gdf['anxiety_score']

    # 5. DATA SERIALIZATION
    # -------------------------------------------------------------------------
    # Saving as Parquet to optimize I/O and preserve metadata for Task 2
    output_path = os.path.join(STORAGE_PATH, "processed_features_1M.parquet")
    gdf.to_parquet(output_path)
    print(f"Data engineering completed. Serialized dataset saved to: {output_path}")

if __name__ == "__main__":
    run_data_engineering_pipeline()

Mounting Google Drive for persistent storage...
Mounted at /content/drive
Initiating data acquisition protocol from Kaggle...
Using Colab cache for faster access to the 'student-mental-health-and-burnout' dataset.
Executing performance benchmarking: CPU (Pandas) vs. GPU (cuDF)...
CPU Processing Latency: 4.9373 seconds
GPU Processing Latency: 0.2279 seconds
Hardware Acceleration Throughput Gain: 21.67x
Commencing high-fidelity feature engineering on 1,000,000 records...
Data engineering completed. Serialized dataset saved to: /content/drive/MyDrive/Colab Notebooks/processed_features_1M.parquet
